In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import time
import numpy as np
import os
import pandas as pd
from collections import defaultdict
from constants import WORD_TYPES, PROPER_NOUN_TYPES
pd.options.display.max_columns = 100
pd.options.display.max_rows = 130

In [3]:
from utils_shared_hanzi_shorts import (
    load_video_configs, draw_vocab_list_whole_image, 
    create_video_with_highlights, create_video_without_highlights
)
from utils_data import (load_raw_data, check_dups)

# Settings

In [ ]:
audio_settings = {
    'voice_name_zh': 'zh-CN-XiaoxiaoNeural',
    'audio_plan': 'c2word',
    'pause_ms_beginning': 150,
    'pause_ms_within_word': 200,
    'pause_ms_between': 500,
}

data_settings = {
    'current_hsk_level': '3.0',
    'priority_limit': 1,
    'sort_cols': ['cat_v3', 'cat3_v3', 'pinyin'],
    'sort_ascending': [True, True, True],
    'words_rmv': ['父亲', '母亲'],
    'output_path': 'output/hsk_top_shorts/',
    'output_path_audio': 'output/hsk_top_shorts/audio/',
    'output_path_images': 'output/hsk_top_shorts/images/',
    'n_words_per_video': 10,
}

video_configs = load_video_configs()
video_configs['words_settings'] = {
    'x': {'chinese': 50, 'pinyin': 165, 'english': 420},
    'max_line_length_buffer_size': {'chinese': 10, 'pinyin': 20, 'english': 60},
    'max_line_length': {},
    'y_gap': 42,
    'spacing': 50,
    'font_size': {'chinese': 36, 'pinyin': 36, 'english': 36},
    'align': {'chinese': 'left', 'pinyin': 'left', 'english': 'left'},
    'fill': {'chinese': '#000000', 'pinyin': '#000000', 'english': '#000000'},
}
video_configs['words_settings']['y'] = video_configs['horizontal_line']['y'] + \
    video_configs['words_settings']['y_gap']
video_configs['words_settings']['max_line_length']['chinese'] = video_configs['words_settings']['x']['pinyin'] - video_configs['words_settings']['x']['chinese'] - video_configs['words_settings']['max_line_length_buffer_size']['chinese']
video_configs['words_settings']['max_line_length']['pinyin'] = video_configs['words_settings']['x']['english'] - video_configs['words_settings']['x']['pinyin'] - video_configs['words_settings']['max_line_length_buffer_size']['pinyin']
video_configs['words_settings']['max_line_length']['english'] = video_configs['bg_size'][0] - video_configs['words_settings']['x']['english'] - video_configs['words_settings']['max_line_length_buffer_size']['english']

video_configs['title_format'] = 'english_only'
video_configs['title_text_settings'] = {
    'text': f"HSK{data_settings['current_hsk_level'][0]}\ntop words",
    'font_path': video_configs['font_path'],
    'font_size': 50,
    'y': 50,
    'spacing': 20,
    'align': 'center',
    'fill': 'black',
    'max_line_length': video_configs['bg_size'][0] - (2 * 50),
}
data_settings['shared_char'] = f"hsk{data_settings['current_hsk_level'][0]}"

# Load data

In [70]:
truly_load_data = False
df_all_vocab = load_raw_data(truly_load_data=truly_load_data)
df_dups = check_dups(df_all_vocab)
print(df_all_vocab.shape)
print(f'# duplicate vocab: {len(df_dups)}')
df_all_vocab.head(3)

!!!!!!!! WARNING: not truly loading data !!!!!!!!
(7600, 38)
# duplicate vocab: 0


,id,chinese,pinyin,english,type,priority,category1,category2,cat_v3,cat2_v3,cat3_v3,hsk_level,known,known_pinyin_prompt,known_english_prompt,quality,word1,word1_english,word2,word2_english,word3,word3_english,word4,word4_english,voice_zh,voice_en,video_notes,sentence,sentence_pinyin,sentence_english,date,source1,source2,funny,per,adu,slang,phonetic
0,152,意见,yì jiàn,opinion,word,1,people,NaN,Language & Expression,NaN,观点与立场,2.0,4.0,2.0,5.0,2.0,意思,meaning,见,to see,NaN,NaN,NaN,NaN,NaN,NaN,NaN,大家都提出了不同意见,Dàjiā dōu tíchū le bùtóng yìjiàn,Everyone gave different opinions,2025-01-02,NaN,NaN,NaN,5.0,5.0,5.0,NaN
1,407,相机,xiàng jī,camera,word,1,hobbies,NaN,Devices & Electronics,NaN,摄影与光学设备,2.0,5.0,2.0,5.0,3.0,相信,to believe,机,machine,NaN,NaN,NaN,NaN,NaN,NaN,NaN,我带了相机拍照,Wǒ dài le xiàngjī pāizhào,I brought a camera to take photos,2025-01-02,NaN,NaN,NaN,5.0,5.0,5.0,NaN
2,749,机会,jī huì,opportunity;chance,word,1,career,Other,Education & Learning,NaN,结果与影响,2.0,2.0,1.0,5.0,3.0,机,machine,会,to be able to,NaN,NaN,NaN,NaN,NaN,NaN,NaN,这次机会很难得,Zhè cì jīhuì hěn nándé,This is a rare opportunity,2025-01-02,NaN,NaN,NaN,5.0,5.0,5.0,NaN


In [71]:
# # Most important HSK1
# current_hsk_level = 6
# df_this_hsk = df_all_vocab[df_all_vocab['hsk_level']==current_hsk_level]
# df_this_hsk['priority'].value_counts()

# Most important HSK3: 62; HSK4: 72; HSK5: 34+99; HSK6: 24+70
# current_hsk_level = 6
# priority_limit = 2
# df_this_hsk = df_all_vocab[
#     (df_all_vocab['hsk_level']==current_hsk_level) &
#     (df_all_vocab['priority']<=priority_limit)
#     ]
# df_this_hsk.sample(10)

In [72]:
def filter_df_for_videos(df_all_vocab, data_settings):
    df_this_hsk = df_all_vocab[
        # (df_all_vocab['hsk_level']==int(data_settings['current_hsk_level'][0])) &
        (df_all_vocab['hsk_level']==data_settings['current_hsk_level']) &
        (df_all_vocab['priority']<=data_settings['priority_limit']) &
        (~df_all_vocab['chinese'].isin(data_settings['words_rmv']))
    ].sort_values(data_settings['sort_cols'], ascending=data_settings['sort_ascending']).reset_index(drop=True)
    return df_this_hsk
    

df_filt = filter_df_for_videos(df_all_vocab, data_settings)
print(df_filt['chinese'].values.tolist())
print(len(df_filt))
df_filt.head(10)

['牛', '建设', '桥', '衬衫', '普遍', '艺术', '工具', '自动', '空调', '设备', '标准', '专业', '去世', '值得', '吵架', '压力', '好奇', '适应', '关注', '理解', '游戏', '保险', '啤酒', '苹果', '香蕉', '糖', '营养', '性别', '搬家', '身份证', '话题', '情况', '类似', '交流', '困难', '各种', '具体', '特色', '通常', '海关', '体验', '错误', '上升', '升', '大约', '至少', '游泳', '取消', '输入', '退出', '技术', '文件', '以来', '卫生间', '组合', '退休', '老板', '创业', '人民币', '长城']
60


,id,chinese,pinyin,english,type,priority,category1,category2,cat_v3,cat2_v3,cat3_v3,hsk_level,known,known_pinyin_prompt,known_english_prompt,quality,word1,word1_english,word2,word2_english,word3,word3_english,word4,word4_english,voice_zh,voice_en,video_notes,sentence,sentence_pinyin,sentence_english,date,source1,source2,funny,per,adu,slang,phonetic
0,5897,牛,niú,cow,word,1,NaN,NaN,Animals,NaN,哺乳动物,3.0,5.0,2.0,1.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,农场里的牛在吃草,nóng chǎng lǐ de niú zài chī cǎo,The cows on the farm are grazing,2025-11-04,codenames,NaN,NaN,5.0,5.0,5.0,NaN
1,6333,建设,jiàn shè,construction;to build,word,1,NaN,NaN,Architecture & Infrastructure,NaN,NaN,3.0,5.0,5.0,2.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,城市建设速度非常快,chéng shì jiàn shè sù dù fēi cháng kuài,The city’s construction is progressing very qu...,2025-12-12,daily add,NaN,NaN,5.0,5.0,5.0,NaN
2,2991,桥,qiáo,bridge,word,1,travel,NaN,Architecture & Infrastructure,NaN,NaN,3.0,1.0,1.0,1.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,我们走过一座桥,wǒ men zǒu guò yī zuò qiáo,we walked across a bridge,2025-02-10,ltl,NaN,NaN,5.0,5.0,5.0,NaN
3,5892,衬衫,chèn shān,shirt,word,1,NaN,NaN,Clothes & Accessories,NaN,上装,3.0,5.0,2.0,2.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,他今天穿了一件蓝色衬衫,tā jīn tiān chuān le yí jiàn lán sè chèn shān,He wore a blue shirt today,2025-11-04,codenames,NaN,NaN,5.0,5.0,5.0,NaN
4,6487,普遍,pǔ biàn,common,word,1,NaN,NaN,Culture & Society,NaN,NaN,3.0,5.0,5.0,2.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,这种现象在城市里很普遍。,zhè zhǒng xiànxiàng zài chéngshì lǐ hěn pǔbiàn,This phenomenon is very common in cities.,2025-12-22,daily add,NaN,NaN,5.0,5.0,5.0,NaN
5,2125,艺术,yì shù,art,word,1,hobbies,NaN,Culture & Society,NaN,NaN,3.0,2.0,1.0,2.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,艺术可以让人们表达情感,Yìshù kěyǐ ràng rénmen biǎodá qínggǎn,Art allows people to express emotions,2025-01-02,NaN,NaN,NaN,5.0,5.0,5.0,NaN
6,6004,工具,gōng jù,tool,word,1,NaN,NaN,Devices & Electronics,NaN,工具与维修设备,3.0,5.0,2.0,2.0,4.0,工作,work,具体,specific,NaN,NaN,NaN,NaN,NaN,NaN,NaN,这些工具要放回工具箱里。,zhè xiē gōng jù yào fàng huí gōng jù xiāng lǐ.,These tools should be put back in the toolbox.,2025-11-04,codenames,NaN,NaN,5.0,5.0,5.0,NaN
7,144,自动,zì dòng,automatic,word,1,electronics,AI & Future Tech,Devices & Electronics,NaN,状态与描述,3.0,1.0,1.0,2.0,1.0,自,self,运动,to move,NaN,NaN,NaN,NaN,NaN,NaN,NaN,灯会在天黑的时候自动亮起,Dēng huì zài tiānhēi de shíhòu zìdòng liàngqǐ,The light turns on automatically when it gets ...,2025-01-02,NaN,NaN,NaN,5.0,5.0,5.0,NaN
8,365,空调,kòng tiáo,air conditioning,word,1,electronics,Appliances & Everyday Electronics,Devices & Electronics,NaN,生活电器与环境调节,3.0,1.0,1.0,1.0,1.0,空气,air,调,adjust,NaN,NaN,NaN,NaN,NaN,NaN,NaN,夏天开空调很舒服,xià tiān kāi kōng tiáo hěn shū fú,turning on the air conditioner in summer is ve...,2025-01-02,NaN,NaN,NaN,5.0,5.0,5.0,NaN
9,867,设备,shè bèi,equipment,word,1,thing,NaN,Devices & Electronics,NaN,电子设备,3.0,5.0,2.0,2.0,3.0,设计,design,准备,preparation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,这个设备很旧了,Zhège shèbèi hěn jiù le,This device is really old,2025-01-02,NaN,NaN,NaN,5.0,5.0,5.0,NaN


In [73]:
from edge_tts import Communicate
import os
voice_name = 'zh-CN-XiaoxiaoNeural'

start_time = time.time()
for row_i, current_row in df_filt.iterrows():
  mp3_path = f"output/tts/{voice_name}/{current_row['chinese']}.mp3"
  if os.path.exists(mp3_path):
     print(f'{mp3_path} exists')
  else:
    print(f"{current_row['chinese']}: {(time.time() - start_time):.2f}")
    communicate = Communicate(current_row['chinese'], voice_name)
    communicate.save_sync(mp3_path)


output/tts/zh-CN-XiaoxiaoNeural/牛.mp3 exists
output/tts/zh-CN-XiaoxiaoNeural/建设.mp3 exists
output/tts/zh-CN-XiaoxiaoNeural/桥.mp3 exists
output/tts/zh-CN-XiaoxiaoNeural/衬衫.mp3 exists
output/tts/zh-CN-XiaoxiaoNeural/普遍.mp3 exists
output/tts/zh-CN-XiaoxiaoNeural/艺术.mp3 exists
output/tts/zh-CN-XiaoxiaoNeural/工具.mp3 exists
output/tts/zh-CN-XiaoxiaoNeural/自动.mp3 exists
output/tts/zh-CN-XiaoxiaoNeural/空调.mp3 exists
output/tts/zh-CN-XiaoxiaoNeural/设备.mp3 exists
output/tts/zh-CN-XiaoxiaoNeural/标准.mp3 exists
output/tts/zh-CN-XiaoxiaoNeural/专业.mp3 exists
output/tts/zh-CN-XiaoxiaoNeural/去世.mp3 exists
output/tts/zh-CN-XiaoxiaoNeural/值得.mp3 exists
output/tts/zh-CN-XiaoxiaoNeural/吵架.mp3 exists
output/tts/zh-CN-XiaoxiaoNeural/压力.mp3 exists
output/tts/zh-CN-XiaoxiaoNeural/好奇.mp3 exists
output/tts/zh-CN-XiaoxiaoNeural/适应.mp3 exists
output/tts/zh-CN-XiaoxiaoNeural/关注.mp3 exists
output/tts/zh-CN-XiaoxiaoNeural/理解.mp3 exists
output/tts/zh-CN-XiaoxiaoNeural/游戏.mp3 exists
output/tts/zh-CN-XiaoxiaoNeural/保险.m

# Create video for each part

In [74]:

from pydub import AudioSegment
def stitch_audios(audio_settings, data_settings, example_words):
    dict_audio_durations = defaultdict(list)
    if audio_settings['audio_plan'] == 'c2word':
        current_start_time = 0

        # beginning pause
        pause_beginning = AudioSegment.silent(duration=audio_settings['pause_ms_beginning'])
        combined = pause_beginning
        dict_audio_durations['audio_path'].append('pause_beginning')
        dict_audio_durations['duration'].append(audio_settings['pause_ms_beginning'] / 1000)
        dict_audio_durations['start_time'].append(current_start_time)
        current_start_time += audio_settings['pause_ms_beginning'] / 1000
        dict_audio_durations['end_time'].append(current_start_time)

        # words
        for _, word in enumerate(example_words):
            # word audio
            word_audio_path = f"output/tts/{audio_settings['voice_name_zh']}/{word}.mp3"
            audio = AudioSegment.from_mp3(word_audio_path)
            combined += audio 
            dict_audio_durations['audio_path'].append(word_audio_path)
            dict_audio_durations['duration'].append(audio.duration_seconds)
            dict_audio_durations['start_time'].append(current_start_time)
            current_start_time += audio.duration_seconds
            dict_audio_durations['end_time'].append(current_start_time)

            # within-word pause
            pause_within_word = AudioSegment.silent(duration=audio_settings['pause_ms_within_word'])
            combined += pause_within_word
            dict_audio_durations['audio_path'].append('within_word_pause')
            dict_audio_durations['duration'].append(audio_settings['pause_ms_within_word'] / 1000)
            dict_audio_durations['start_time'].append(current_start_time)
            current_start_time += audio_settings['pause_ms_within_word'] / 1000
            dict_audio_durations['end_time'].append(current_start_time)

            # word again audio
            combined += audio 
            dict_audio_durations['audio_path'].append(word_audio_path)
            dict_audio_durations['duration'].append(audio.duration_seconds)
            dict_audio_durations['start_time'].append(current_start_time)
            current_start_time += audio.duration_seconds
            dict_audio_durations['end_time'].append(current_start_time)

            # inter-word pause
            pause_inter_word = AudioSegment.silent(duration=audio_settings['pause_ms_between'])
            combined += pause_inter_word
            dict_audio_durations['audio_path'].append('inter_word_pause')
            dict_audio_durations['duration'].append(audio_settings['pause_ms_between'] / 1000)
            dict_audio_durations['start_time'].append(current_start_time)
            current_start_time += audio_settings['pause_ms_between'] / 1000
            dict_audio_durations['end_time'].append(current_start_time)
            

    # export the combined audio file
    combined.export(f"{data_settings['output_path_audio']}/!combined_part{data_settings['current_part']}.mp3", format="mp3")
    print(f'Audio duration: {combined.duration_seconds:.1f}s')

    # Add in static slide audio into dataframe of audio durations
    df_durations = pd.DataFrame(dict_audio_durations)
    return df_durations

In [75]:
print('Cut vocab into parts')
data_settings['n_words_total'] = len(df_filt)
data_settings['n_parts'] = int(np.ceil(len(df_filt) / data_settings['n_words_per_video']))

for current_part in range(1, data_settings['n_parts'] + 1):
    # Determine vocabulary in current part
    data_settings['current_part'] = current_part
    start_index = (current_part - 1) * data_settings['n_words_per_video']
    end_index = start_index + data_settings['n_words_per_video'] - 1
    data_settings['current_part_index_range'] = (start_index, end_index)
    print(f"Processing part {current_part}/{data_settings['n_parts']} with index range {data_settings['current_part_index_range']}")
    df_filt_currentpart = df_filt[
        (df_filt.index >= data_settings['current_part_index_range'][0]) &
        (df_filt.index <= data_settings['current_part_index_range'][1])
    ].reset_index(drop=True)

    print('Making audio')
    df_durations = stitch_audios(audio_settings, data_settings, df_filt_currentpart['chinese'].values.tolist())

    print('Making image')
    no_hl_img_file_path = draw_vocab_list_whole_image(video_configs, data_settings, df_filt_currentpart, title_format=video_configs['title_format'])
    print('Making video without highlights')
    create_video_without_highlights(data_settings, video_configs, no_hl_img_file_path)
    print('Making video with highlights')
    create_video_with_highlights(df_durations, audio_settings, data_settings, video_configs, method_to_determine_highlight_timing=None, highlight_start_ids=np.arange(1,len(df_durations), 4))
    break
    

Cut vocab into parts
Processing part 1/6 with index range (0, 9)
Making audio


frame_index:  34%|███▍      | 254/742 [00:10<00:09, 50.84it/s, now=None]

Audio duration: 30.9s
Making image
Making video without highlights
MoviePy - Building video output/hsk_top_shorts//hsk3_no_highlights_part1.mp4.
MoviePy - Writing audio in hsk3_no_highlights_part1TEMP_MPY_wvf_snd.mp3


frame_index:  34%|███▍      | 254/742 [00:10<00:09, 50.84it/s, now=None]

MoviePy - Done.
MoviePy - Writing video output/hsk_top_shorts//hsk3_no_highlights_part1.mp4



frame_index:  34%|███▍      | 254/742 [00:24<00:09, 50.84it/s, now=None]

MoviePy - Done !
MoviePy - video ready output/hsk_top_shorts//hsk3_no_highlights_part1.mp4
Making video with highlights
MoviePy - Building video output/hsk_top_shorts//hsk3_part1.mp4.
MoviePy - Writing audio in hsk3_part1TEMP_MPY_wvf_snd.mp3


frame_index:  34%|███▍      | 254/742 [00:24<00:09, 50.84it/s, now=None]

MoviePy - Done.
MoviePy - Writing video output/hsk_top_shorts//hsk3_part1.mp4



KeyboardInterrupt: 

In [60]:
df_filt

,id,chinese,pinyin,english,type,priority,category1,category2,cat_v3,cat2_v3,cat3_v3,hsk_level,known,known_pinyin_prompt,known_english_prompt,quality,word1,word1_english,word2,word2_english,word3,word3_english,word4,word4_english,voice_zh,voice_en,video_notes,sentence,sentence_pinyin,sentence_english,date,source1,source2,funny,per,adu,slang,phonetic
0,5897,牛,niú,cow,word,1,NaN,NaN,Animals,NaN,哺乳动物,3.0,5.0,2.0,1.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,农场里的牛在吃草,nóng chǎng lǐ de niú zài chī cǎo,The cows on the farm are grazing,2025-11-04,codenames,NaN,NaN,5.0,5.0,5.0,NaN
1,6333,建设,jiàn shè,construction;to build,word,1,NaN,NaN,Architecture & Infrastructure,NaN,NaN,3.0,5.0,5.0,2.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,城市建设速度非常快,chéng shì jiàn shè sù dù fēi cháng kuài,The city’s construction is progressing very qu...,2025-12-12,daily add,NaN,NaN,5.0,5.0,5.0,NaN
2,2991,桥,qiáo,bridge,word,1,travel,NaN,Architecture & Infrastructure,NaN,NaN,3.0,1.0,1.0,1.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,我们走过一座桥,wǒ men zǒu guò yī zuò qiáo,we walked across a bridge,2025-02-10,ltl,NaN,NaN,5.0,5.0,5.0,NaN
3,5892,衬衫,chèn shān,shirt,word,1,NaN,NaN,Clothes & Accessories,NaN,上装,3.0,5.0,2.0,2.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,他今天穿了一件蓝色衬衫,tā jīn tiān chuān le yí jiàn lán sè chèn shān,He wore a blue shirt today,2025-11-04,codenames,NaN,NaN,5.0,5.0,5.0,NaN
4,6487,普遍,pǔ biàn,common,word,1,NaN,NaN,Culture & Society,NaN,NaN,3.0,5.0,5.0,2.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,这种现象在城市里很普遍。,zhè zhǒng xiànxiàng zài chéngshì lǐ hěn pǔbiàn,This phenomenon is very common in cities.,2025-12-22,daily add,NaN,NaN,5.0,5.0,5.0,NaN
5,2125,艺术,yì shù,art,word,1,hobbies,NaN,Culture & Society,NaN,NaN,3.0,2.0,1.0,2.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,艺术可以让人们表达情感,Yìshù kěyǐ ràng rénmen biǎodá qínggǎn,Art allows people to express emotions,2025-01-02,NaN,NaN,NaN,5.0,5.0,5.0,NaN
6,6004,工具,gōng jù,tool,word,1,NaN,NaN,Devices & Electronics,NaN,工具与维修设备,3.0,5.0,2.0,2.0,4.0,工作,work,具体,specific,NaN,NaN,NaN,NaN,NaN,NaN,NaN,这些工具要放回工具箱里。,zhè xiē gōng jù yào fàng huí gōng jù xiāng lǐ.,These tools should be put back in the toolbox.,2025-11-04,codenames,NaN,NaN,5.0,5.0,5.0,NaN
7,144,自动,zì dòng,automatic,word,1,electronics,AI & Future Tech,Devices & Electronics,NaN,状态与描述,3.0,1.0,1.0,2.0,1.0,自,self,运动,to move,NaN,NaN,NaN,NaN,NaN,NaN,NaN,灯会在天黑的时候自动亮起,Dēng huì zài tiānhēi de shíhòu zìdòng liàngqǐ,The light turns on automatically when it gets ...,2025-01-02,NaN,NaN,NaN,5.0,5.0,5.0,NaN
8,365,空调,kòng tiáo,air conditioning,word,1,electronics,Appliances & Everyday Electronics,Devices & Electronics,NaN,生活电器与环境调节,3.0,1.0,1.0,1.0,1.0,空气,air,调,adjust,NaN,NaN,NaN,NaN,NaN,NaN,NaN,夏天开空调很舒服,xià tiān kāi kōng tiáo hěn shū fú,turning on the air conditioner in summer is ve...,2025-01-02,NaN,NaN,NaN,5.0,5.0,5.0,NaN
9,867,设备,shè bèi,equipment,word,1,thing,NaN,Devices & Electronics,NaN,电子设备,3.0,5.0,2.0,2.0,3.0,设计,design,准备,preparation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,这个设备很旧了,Zhège shèbèi hěn jiù le,This device is really old,2025-01-02,NaN,NaN,NaN,5.0,5.0,5.0,NaN


In [61]:
df_durations.head(10)

,audio_path,duration,start_time,end_time
0,pause_beginning,0.150,0.000,0.150
1,output/tts/zh-CN-XiaoxiaoNeural/牛.mp3,0.984,0.150,1.134
2,within_word_pause,0.200,1.134,1.334
3,output/tts/zh-CN-XiaoxiaoNeural/牛.mp3,0.984,1.334,2.318
4,inter_word_pause,0.500,2.318,2.818
5,output/tts/zh-CN-XiaoxiaoNeural/建设.mp3,1.296,2.818,4.114
6,within_word_pause,0.200,4.114,4.314
7,output/tts/zh-CN-XiaoxiaoNeural/建设.mp3,1.296,4.314,5.610
8,inter_word_pause,0.500,5.610,6.110
9,output/tts/zh-CN-XiaoxiaoNeural/桥.mp3,1.056,6.110,7.166
